# Step 1 - Sensitivity and smoothness of the deterministic map

The ARC White-Box challenge asks us to predict a network's mean post-ReLU
activation **from its weights alone**. Once the weights $W$ are fixed the input
$x\sim N(0,I)$ is integrated out, so the target is a *deterministic* function of
the weights:

$$ F(W) \;=\; \mathbb{E}_{x\sim N(0,I)}\big[f(x;W)\big], \qquad
   f(x;W)=\mathrm{ReLU}\big(W_k\cdots\mathrm{ReLU}(W_1 x)\big). $$

Here $W\in\mathbb{R}^{k\times d\times d}$, so $x=\mathrm{flat}(W)\in\mathbb{R}^{kd^2}$
and $F:\mathbb{R}^{kd^2}\to\mathbb{R}^{d}$. We use $d=8$, $k=8$ (see
`whest_toy.py` for why width 2 is degenerate).

**What this notebook probes.** $f$ is only $C^0$ in $W$ (ReLU gives kinks), and I
suspect that jaggedness controls how hard $F$ is to learn. The three studies below:

1. **Money plot** - Frobenius norm of $\partial f/\partial W_i$ vs. layer index
   $i$: how sensitivity depends on a layer's distance from the output.
2. **Interactive line-scan** - scrub a perturbation $W+t\,\Delta W$ and watch $f$
   kink while $F=\mathbb{E}_x[f]$ stays smooth. *This is the payoff:* averaging
   over $x$ integrates the measure-zero kinks away, which is why an MLP can hope
   to fit $F$ (step 2).
3. **Label-convergence** - how the Monte-Carlo estimate of $F$ converges with the
   sample count $M$, pinning down the real noise constant $\sigma_f$ (not an
   assumed one) for step 2's label budget.

In [1]:
import numpy as np
import torch
from torch.func import jacrev, vmap
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider, FloatSlider

import whest_toy as toy

torch.set_default_dtype(torch.float64)  # step 1 is analysis: prefer clean derivatives
SEED = 0
K, D = toy.K, toy.D          # depth, width  (8, 8)
print(f"geometry: depth k={K}, width d={D},  W in R^{K*D*D},  F: R^{K*D*D} -> R^{D}")

geometry: depth k=8, width d=8,  W in R^512,  F: R^512 -> R^8


## Sanity check: the network is alive

At width 8 the net is not dead - some inputs survive all $k$ layers, so
$F(W)\not\equiv 0$. (At width 2 every one of these would be identically zero.)

In [2]:
W = toy.sample_weights(SEED)                     # (k, d, d)
X = torch.randn(4096, D, generator=torch.Generator().manual_seed(1))
Z = toy.f(W, X)                                  # (4096, d) post-ReLU final layer
print("alive fraction (inputs with nonzero output):", float((Z.abs().sum(1) > 0).float().mean()))
print("F(W) via MC:", toy.F_mc(W, 8192, 1).numpy().round(4))
print("all outputs >= 0 (F is a mean of nonnegatives):", bool((Z >= 0).all()))

alive fraction (inputs with nonzero output): 0.83203125
F(W) via MC: [0.     0.0467 0.0009 0.0512 0.0225 0.0005 0.0122 0.0001]
all outputs >= 0 (F is a mean of nonnegatives): True


## 1. Money plot: sensitivity vs. layer index

For a single input $x$, $J(x)=\partial f(x;W)/\partial W$ is a
$(d_{\text{out}}, k, d, d)$ tensor. The per-layer sensitivity is the Frobenius
norm of its layer-$i$ block, $\lVert \partial f/\partial W_i\rVert_F$.

The **jagged** quantity is the single-$x$ sensitivity (it depends on which ReLU
gates $x$ happens to open). The **smooth** quantity is the sensitivity of the
*averaged* map, $\lVert\partial F/\partial W_i\rVert_F$ with
$\partial F/\partial W = \mathbb{E}_x[J(x)]$ - we average the Jacobian first,
then take the norm. Watch how both depend on the layer's distance from the
output ($i=k$ is the output layer).

In [3]:
# per-input Jacobian of f w.r.t. W, batched over x with vmap
def f_single(W, x):                              # x: (d,) -> out: (d,)
    return toy.f(W, x.unsqueeze(0)).squeeze(0)

J_of = jacrev(f_single, argnums=0)               # (W, x) -> (d_out, k, d, d)
Xj = torch.randn(2000, D, generator=torch.Generator().manual_seed(2))
J = vmap(lambda x: J_of(W, x))(Xj)               # (n, d_out, k, d, d)

# Frobenius norm of each layer's block: norm over (d_out, d, d) for each layer i
sens_f = J.permute(0, 2, 1, 3, 4).reshape(J.shape[0], K, -1).norm(dim=2)  # (n, k)

# smooth version: average the Jacobian over x, THEN norm per layer
J_F = J.mean(0)                                  # (d_out, k, d, d) = dF/dW
sens_F = J_F.permute(1, 0, 2, 3).reshape(K, -1).norm(dim=1)    # (k,)

layers = np.arange(1, K + 1)
print("mean single-x sensitivity per layer:", sens_f.mean(0).numpy().round(3))
print("averaged-map sensitivity  per layer:", sens_F.numpy().round(3))

mean single-x sensitivity per layer: [0.278 0.362 0.468 0.252 0.484 0.784 0.664 0.159]
averaged-map sensitivity  per layer: [0.049 0.113 0.244 0.155 0.329 0.669 0.586 0.143]


In [4]:
lo = torch.quantile(sens_f, 0.1, dim=0).numpy()
hi = torch.quantile(sens_f, 0.9, dim=0).numpy()
med = sens_f.median(0).values.numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[layers, layers[::-1]], y=np.r_[hi, lo[::-1]],
                         fill="toself", fillcolor="rgba(99,110,250,0.15)",
                         line=dict(width=0), name="single-x  10-90%", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=layers, y=med, mode="lines+markers",
                         line=dict(color="rgba(99,110,250,0.9)"),
                         name="single-x  median  (jagged, input-dependent)"))
fig.add_trace(go.Scatter(x=layers, y=sens_F.numpy(), mode="lines+markers",
                         line=dict(color="crimson", width=3),
                         name="||dF/dW_i||  (averaged, smooth)"))
fig.update_layout(title="Money plot: sensitivity of f / F to each layer's weights",
                  xaxis_title="layer index i  (i=k is the output layer)",
                  yaxis_title="Frobenius sensitivity  ||d(.)/dW_i||_F",
                  template="plotly_white", height=430)
fig.show()

**Read it:** the crimson curve $\lVert\partial F/\partial W_i\rVert$ sits *below*
the single-$x$ band - averaging over $x$ partially cancels the input-dependent
Jacobians (triangle inequality: $\lVert\mathbb{E}[J]\rVert\le\mathbb{E}\lVert
J\rVert$). The shape vs. $i$ tells you whether early layers (far from the output,
their perturbations pass through many downstream ReLU gates) matter more or less
than the output layer.

## 2. Interactive line-scan: f kinks, F is smooth

Perturb the weights along a fixed unit direction, $W(t)=W+t\,\Delta W$, and plot
one output coordinate as $t$ varies.

- **Thin lines** = $f(x;W(t))$ for a few *fixed* inputs $x$. Each is
  **piecewise-linear with kinks** - a kink is a $t$ where some ReLU flips sign.
- **Bold line** = $F(W(t))=\mathbb{E}_x[f]$ (Monte-Carlo). The kinks of different
  $x$ land at different $t$, so averaging **smooths them out**: $F$ is $C^1$
  (generically $C^\infty$) even though every $f$ is only $C^0$.

Scrub the sliders. `layer` chooses which layer's weights to perturb ($-1$ = all
layers jointly); `coord` chooses the output neuron; `t_max` the scan half-width.

In [5]:
def line_scan(w_seed=0, layer=-1, coord=0, x_seed=3, dir_seed=4, t_max=3.0):
    W0 = toy.sample_weights(w_seed)                         # (k, d, d)

    # unit-Frobenius perturbation direction, optionally confined to one layer
    g = torch.Generator().manual_seed(dir_seed)
    dW = torch.randn(K, D, D, generator=g)
    if layer >= 0:
        mask = torch.zeros(K, D, D); mask[layer] = 1.0
        dW = dW * mask
    dW = dW / dW.norm()                                     # ||dW||_F = 1

    ts = torch.linspace(-t_max, t_max, 400)                 # (T,)
    x_few = torch.randn(5, D, generator=torch.Generator().manual_seed(x_seed))   # fixed inputs
    x_mc  = torch.randn(4000, D, generator=torch.Generator().manual_seed(x_seed + 100))

    f_curves = torch.empty(len(ts), len(x_few))             # (T, 5)  single-x
    F_curve  = torch.empty(len(ts))                         # (T,)    the mean
    for j, t in enumerate(ts):
        Wt = W0 + t * dW
        f_curves[j] = toy.f(Wt, x_few)[:, coord]
        F_curve[j]  = toy.f(Wt, x_mc)[:, coord].mean()

    fig = go.Figure()
    for m in range(x_few.shape[0]):
        fig.add_trace(go.Scatter(x=ts.numpy(), y=f_curves[:, m].numpy(), mode="lines",
                                 line=dict(width=1), opacity=0.6,
                                 name=f"f(x_{m})", showlegend=(m == 0)))
    fig.add_trace(go.Scatter(x=ts.numpy(), y=F_curve.numpy(), mode="lines",
                             line=dict(color="crimson", width=3), name="F = E_x[f]  (smooth)"))
    tag = "all layers" if layer < 0 else f"layer {layer}"
    fig.update_layout(title=f"Line-scan along W + t*dW  ({tag}),  output coord {coord}",
                      xaxis_title="t  (perturbation magnitude, ||dW||_F = 1)",
                      yaxis_title=f"output coordinate {coord}",
                      template="plotly_white", height=430)
    fig.show()

interact(line_scan,
         w_seed=IntSlider(value=0, min=0, max=20), layer=IntSlider(value=-1, min=-1, max=K - 1),
         coord=IntSlider(value=0, min=0, max=D - 1), x_seed=IntSlider(value=3, min=0, max=20),
         dir_seed=IntSlider(value=4, min=0, max=20),
         t_max=FloatSlider(value=3.0, min=0.5, max=8.0, step=0.5));

interactive(children=(IntSlider(value=0, description='w_seed', max=20), IntSlider(value=-1, description='layer…

## 3. Label-convergence: how clean is a Monte-Carlo label?

Step 2 needs labels $F(W)=\mathbb{E}_x[f]$, gotten by averaging $M$ samples. The
error of the sample mean is $\sigma_f/\sqrt{M}$ per output coordinate, where
$\sigma_f=\mathrm{std}_x[f(x;W)]$. The rate $1/\sqrt{M}$ is guaranteed by the CLT
(ReLU does not break it), but the **constant $\sigma_f$ is unknown** - and if a
neuron is mostly dead, its *relative* error can be large. We measure $\sigma_f$
rather than assume it, and check the slope really is $-1/2$ (a guard against
heavy tails hiding in the ReLU composition).

In [6]:
# reference F at huge M, and the per-coordinate sigma_f, for a few networks
Ms = np.unique(np.round(np.logspace(1, 4.7, 14)).astype(int))     # ~10 .. ~50000
w_seeds = range(6)
big = 200_000
err = np.zeros((len(w_seeds), len(Ms)))                            # ||F_hat(M) - F_ref||
sigma_f = {}
for r, ws in enumerate(w_seeds):
    Wr = toy.sample_weights(ws)
    Xbig = torch.randn(big, D, generator=torch.Generator().manual_seed(500 + ws))
    Zbig = toy.f(Wr, Xbig)                                         # (big, d)
    F_ref = Zbig.mean(0)                                           # ~exact target
    sigma_f[ws] = Zbig.std(0)                                      # (d,) per-coordinate noise
    for c, M in enumerate(Ms):
        F_hat = Zbig[:M].mean(0)                                   # first M as an M-sample estimate
        err[r, c] = float((F_hat - F_ref).norm())

sig0 = sigma_f[0]
print("sigma_f per coordinate (net 0):", sig0.numpy().round(3))
print("mean |F| (net 0):", toy.F_mc(toy.sample_weights(0), big, 500).abs().mean().item())
print("=> at M=8192, label std ~ sigma_f/sqrt(M) ~", float(sig0.mean() / np.sqrt(8192)))

sigma_f per coordinate (net 0): [0.    0.055 0.004 0.057 0.03  0.005 0.019 0.002]
mean |F| (net 0): 0.016640948716455342
=> at M=8192, label std ~ sigma_f/sqrt(M) ~ 0.00023767481307998352


In [7]:
theory = float(sig0.norm()) / np.sqrt(Ms.astype(float))          # ||sigma_f||/sqrt(M): expected error
fig = go.Figure()
for r, ws in enumerate(w_seeds):
    fig.add_trace(go.Scatter(x=Ms, y=err[r], mode="lines+markers",
                             opacity=0.6, name=f"net {ws}"))
fig.add_trace(go.Scatter(x=Ms, y=theory, mode="lines",
                         line=dict(color="black", dash="dash", width=2),
                         name="||sigma_f|| / sqrt(M)  (theory, slope -1/2)"))
fig.update_layout(title="MC label error vs sample count M",
                  xaxis_title="M (samples per label)", yaxis_title="|| F_hat(M) - F_ref ||",
                  xaxis_type="log", yaxis_type="log", template="plotly_white", height=430)
fig.show()

**Takeaway for step 2.** The measured $\sigma_f$ (not an assumed one) sets the
label-noise floor $\sigma_f/\sqrt{M}$. With the value printed above we pick $M$
so that label noise is well below any error we would call "the MLP failing," and
we can attribute a failure in step 2 to coverage/model ($N$, capacity) rather
than noisy labels.